## Limpieza de 'DF_ITER5_SUCIO.xlsx'

Cargamos la tabla y le hacemos una primera inspección antes de limpiarla, siguiendo el mismo proceso que en buscametas/xipgroc/carreirasgalegas/ccnorte/cronofinisher/mychip/sportmaniacs/cursescat.

A diferencia de las demás fuentes, aquí cada carrera+modalidad viene repartida en **dos filas** (una por `sexe`, F/M) en vez de en columnas separadas — el primer paso es pivotar para tener `finisher_d`/`finisher_h` en la misma fila, igual que hicimos con `en_meta_d`/`en_meta_h` en mychip pero aquí hay que reconstruirlo desde cero. `tipus` es un vocabulario cerrado y limpio (18 valores: RUN_10, TRAIL, BTT, MARXA...), así que se mapea directamente en vez de por palabras clave.</cell id="cell-0">


In [1]:
from pathlib import Path
import pandas as pd

XLSX_PATH = Path("../../data/raw/iter5/DF_ITER5_SUCIO.xlsx")
curses = pd.read_excel(XLSX_PATH, sheet_name="mmm")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (3321, 6)

nom                 object
poblacio            object
data        datetime64[ns]
tipus               object
sexe                object
arribats             int64
dtype: object


,nom,poblacio,data,tipus,sexe,arribats
0,10K Lleida International Race,LLEIDA,2024-11-24,RUN_10,F,92
1,10K Lleida International Race,LLEIDA,2024-11-24,RUN_10,M,376
2,AECC en marxa - Mollerussa - 5k,MOLLERUSSA,2018-05-13,MARXA,F,552
3,AECC en marxa - Mollerussa - 5k,MOLLERUSSA,2018-05-13,MARXA,M,120
4,AECC en marxa - Mollerussa - 5k,MOLLERUSSA,2019-05-12,MARXA,F,407


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados, valores de "sexe"/"tipus"
# y rango de fechas. No hay ninguna columna con nulos (la fuente ya viene
# muy limpia), así que el foco aquí es entender la estructura fila-por-
# género antes de pivotar.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print()

print("sexe value_counts:")
print(curses["sexe"].value_counts())
print()

print("tipus value_counts:")
print(curses["tipus"].value_counts())
print()

print("Rango de fechas:", curses["data"].min(), "->", curses["data"].max())
print()

_grupos = curses.groupby(["nom", "poblacio", "data", "tipus"])["sexe"].nunique()
print("Grupos carrera+modalidad con las 2 filas de género (F y M):", (_grupos == 2).sum(), "de", len(_grupos))
print("Grupos con solo 1 fila (el otro género tuvo 0 finishers y no se guardó):", (_grupos == 1).sum())

Valores nulos por columna:
nom         0
poblacio    0
data        0
tipus       0
sexe        0
arribats    0
dtype: int64

Filas completamente duplicadas: 0

sexe value_counts:
sexe
M    1687
F    1634
Name: count, dtype: int64

tipus value_counts:
tipus
RUN_10            1132
RUN_5              942
TRAIL              408
CROS               223
BTT                172
RUN_MM             128
MILLA               88
DUATLO              49
INFANTILS           47
TRAVESSIA           40
RUN_3               20
CADIRES             14
VIRTUAL             12
MARXA_ATLETICA      11
MARXA               10
MARATO               9
OBSTACLES            8
RUN_1                4
COTXETS              4
Name: count, dtype: int64

Rango de fechas: 2010-12-31 00:00:00 -> 2026-06-20 00:00:00

Grupos carrera+modalidad con las 2 filas de género (F y M): 1487 de 1834
Grupos con solo 1 fila (el otro género tuvo 0 finishers y no se guardó): 347


In [3]:
# Quitamos duplicados exactos ANTES de pivotar (red de seguridad, aunque
# el diagnóstico de arriba ya confirma 0 duplicados). Después pivotamos:
# cada combinación (nom, poblacio, data, tipus) pasa a ser una sola fila,
# con "finisher_d" (F) y "finisher_h" (M) como columnas — los grupos que
# solo tenían una fila de género quedan a 0 en el género que faltaba
# (fill_value=0), que es lo correcto (0 finishers de ese género).
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

curses_limpio = curses.pivot_table(
    index=["nom", "poblacio", "data", "tipus"],
    columns="sexe", values="arribats", aggfunc="sum", fill_value=0,
).reset_index()
curses_limpio.columns.name = None

curses_limpio = curses_limpio.rename(columns={
    "nom": "nombre_carrera",
    "poblacio": "municipio",
    "data": "fecha",
    "tipus": "modalidad",
    "F": "finisher_d",
    "M": "finisher_h",
})

print(curses_limpio.shape)
curses_limpio.head()

0 filas duplicadas eliminadas (3321 -> 3321)
(1834, 6)


,nombre_carrera,municipio,fecha,modalidad,finisher_d,finisher_h
0,10K Lleida International Race,LLEIDA,2024-11-24,RUN_10,92,376
1,AECC en marxa - Mollerussa - 5k,MOLLERUSSA,2018-05-13,MARXA,552,120
2,AECC en marxa - Mollerussa - 5k,MOLLERUSSA,2019-05-12,MARXA,407,141
3,Aeroport - Alguaire - 10K,ALGUAIRE,2013-10-12,RUN_10,54,294
4,Aeroport - Alguaire - 10K,ALGUAIRE,2014-10-12,RUN_10,30,122


In [4]:
# "modalidad" (antes "tipus") ya viene dada por un código cerrado de 18
# valores (no hace falta clasificar por palabras clave, solo mapear, igual
# que "esport" en cronofinisher/mychip). "CADIRES" (sillas de ruedas) y
# "COTXETS" (cochecitos/carritos de bebé — categoría familiar no
# competitiva) son categorías especiales ajenas a nuestras disciplinas,
# así que van a "Otros" igual que discapacidad/handbike en el resto de
# fuentes. "TRAVESSIA" (travesía a nado) y "OBSTACLES" también van a
# "Otros" por el mismo motivo que en sportmaniacs/cursescat. "VIRTUAL" no
# es una disciplina real sino un formato (carrera a distancia), así que
# tampoco encaja en ninguna categoría. "INFANTILS" no dice la disciplina
# (son carreras de niños de cualquier tipo), así que asumimos road running
# por defecto, la más habitual en estos casos.
_TIPUS_A_TIPO = {
    "RUN_10": "road running", "RUN_5": "road running", "RUN_3": "road running",
    "RUN_1": "road running", "RUN_MM": "road running", "MARATO": "road running",
    "MILLA": "road running", "CROS": "road running", "INFANTILS": "road running",
    "TRAIL": "trail running",
    "BTT": "Ciclismo y btt",
    "DUATLO": "Multidisciplina",
    "MARXA": "marcha", "MARXA_ATLETICA": "marcha",
    "TRAVESSIA": "Otros", "OBSTACLES": "Otros", "CADIRES": "Otros",
    "COTXETS": "Otros", "VIRTUAL": "Otros",
}
curses_limpio["tipo_modalidad"] = curses_limpio["modalidad"].map(_TIPUS_A_TIPO)

print(curses_limpio["tipo_modalidad"].value_counts())
print("Sin mapear (debería ser 0):", curses_limpio["tipo_modalidad"].isna().sum())

tipo_modalidad
road running       1452
trail running       205
Ciclismo y btt       90
Otros                43
Multidisciplina      28
marcha               16
Name: count, dtype: int64
Sin mapear (debería ser 0): 0


In [5]:
# "modalidad" tampoco trae ninguna categoría de edad explícita salvo
# "INFANTILS" — y las dos categorías especiales ("CADIRES"/"COTXETS") van
# a "Otros" (no son una cuestión de edad). Todo lo demás por defecto es
# Absoluta/General, ya que aquí no hay ningún texto de categoría (SUB/
# veterano/élite...) del que extraer más señal.
_TIPUS_A_PUBLICO = {
    "INFANTILS": "Infantil",
    "CADIRES": "Otros",
    "COTXETS": "Otros",
}
_EQUIPOS_PATRON = r"equips?\b|equipos?\b"

# El código de "modalidad" no distingue equipos (no existe ese TIPUS), así
# que la única pista posible es el nombre de la carrera.
curses_limpio["publico"] = curses_limpio["modalidad"].map(_TIPUS_A_PUBLICO).fillna("Absoluta/General")
_es_equipos = curses_limpio["nombre_carrera"].str.contains(_EQUIPOS_PATRON, case=False, regex=True, na=False)
curses_limpio.loc[_es_equipos, "publico"] = "Equipos"

print(curses_limpio["publico"].value_counts())

publico
Absoluta/General    1785
Infantil              36
Otros                 11
Equipos                2
Name: count, dtype: int64


In [6]:
# "modalidad" también nos da la distancia oficial directamente para los
# códigos RUN_X/MARATO/MILLA — el resto (TRAIL/BTT/CROS/MARXA/DUATLO...)
# no llevan ninguna distancia fija (varía de edición a edición), así que
# se quedan en 0, igual que en el resto de fuentes cuando no hay forma
# fiable de saberla.
_TIPUS_A_KM = {
    "RUN_10": 10.0, "RUN_5": 5.0, "RUN_3": 3.0, "RUN_1": 1.0,
    "RUN_MM": 21.097, "MARATO": 42.195, "MILLA": 1.609,
}
curses_limpio["distancia"] = curses_limpio["modalidad"].map(_TIPUS_A_KM).fillna(0.0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))

Filas con distancia detectada: 1206 de 1834


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("iter5") para identificar el origen al concatenar las tablas. `municipio` se recupera de `poblacio`; `comarca`/`provincia` no vienen en la fuente, así que las geocodificamos a partir de `municipio` (igual que en carreirasgalegas/cursescat/cruzandolameta), asumiendo Catalunya para desambiguar (iter5 es una plataforma centrada en Lleida/Ponent). Lo que es propio solo de iter5 (`modalidad`, el código original de `tipus`) va al final, para poder revisar a ojo la clasificación.

In [7]:
# Geocodificamos "comarca"/"provincia" a partir de "municipio" (igual que
# en carreirasgalegas/cursescat). Solo ~100 municipios únicos, así que es
# rápido. Checkpoint propio en iter5_ubicaciones.csv. Asumimos Catalunya
# para desambiguar (iter5 es una plataforma de carreras del Ponent/Lleida,
# aunque algún municipio limítrofe pueda caer en Aragón).
import csv
import time


def geocodificar_ubicacion_municipios(municipios, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "iter5_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["municipio"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="iter5_ubicaciones_claudia")

    municipios_unicos = list(dict.fromkeys(m for m in municipios if isinstance(m, str)))
    pendientes = [m for m in municipios_unicos if m not in cache]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(municipios_unicos)} únicos)")

    campos = ["municipio", "comarca", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, municipio in enumerate(pendientes, 1):
            fila = {"municipio": municipio, "comarca": None, "provincia": None}
            try:
                loc = geolocator.geocode(
                    f"{municipio}, Catalunya, España", exactly_one=True, country_codes="es",
                    addressdetails=True, timeout=10,
                )
                if loc:
                    addr = loc.raw.get("address", {})
                    fila["comarca"] = addr.get("county")
                    fila["provincia"] = addr.get("province") or addr.get("state")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[municipio] = fila

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/iter5")
_ubicaciones = geocodificar_ubicacion_municipios(curses_limpio["municipio"], out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("comarca"))
curses_limpio["provincia"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("provincia"))

print("Filas con provincia:", curses_limpio["provincia"].notna().sum(), "de", len(curses_limpio))
print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "comarca", "provincia"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 101 municipios ya geocodificados
Municipios a geocodificar: 0 (de 101 únicos)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\iter5_data\iter5_ubicaciones.csv
Filas con provincia: 1783 de 1834
Filas con comarca: 1783 de 1834


,municipio,comarca,provincia
242,SOLSONA,Solsonès,Lleida
1512,ARTESA,Noguera,Lleida
3,ALGUAIRE,Segrià,Lleida
754,MASSOTERES,Segarra,Lleida
1323,LES BORGES BLANQUES,Garrigues,Lleida
1645,GUIMERÀ,Urgell,Lleida
169,BALAGUER,Noguera,Lleida
1280,MONTOLIU,Baix Camp,Tarragona
743,CASTELLDANS,Garrigues,Lleida
1689,ALCOLETGE,Segrià,Lleida


In [8]:
# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 8 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 9 fuentes, dejando lo propio de iter5 (modalidad) al final.
curses_limpio["fuente"] = "iter5"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia", "modalidad"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'modalidad']

In [9]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'modalidad']
Filas x columnas: (1834, 13)

fuente                    object
nombre_carrera            object
fecha             datetime64[ns]
dia_semana                object
distancia                float64
tipo_modalidad            object
publico                   object
finisher_d                 int64
finisher_h                 int64
municipio                 object
comarca                   object
provincia                 object
modalidad                 object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Equipos  Infantil  Otros
tipo_modalidad                                             
Ciclismo y btt                 90        0         0      0
Multidisciplina                28        0         0      0
Otros                          30        2         0     11
marcha      

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,modalidad
1,iter5,AECC en marxa - Mollerussa - 5k,2018-05-13,Domingo,0.000,marcha,Absoluta/General,552,120,MOLLERUSSA,Pla d'Urgell,Lleida,MARXA
279,iter5,Cros - Raimat - CAD F,2022-11-06,Domingo,0.000,road running,Absoluta/General,3,0,RAIMAT,Segrià,Lleida,CROS
510,iter5,"Cursa Trenkacames - Rosselló - 9,5K",2017-07-15,Sábado,0.000,trail running,Absoluta/General,8,27,ROSSELLO,Segrià,Lleida,TRAIL
506,iter5,Cursa Trenkacames - Rosselló - 7K,2016-04-17,Domingo,0.000,trail running,Absoluta/General,21,23,ROSSELLÓ,Segrià,Lleida,TRAIL
1404,iter5,SICORIS - Duatló,2018-04-29,Domingo,0.000,Multidisciplina,Absoluta/General,10,82,LLEIDA,Segrià,Lleida,DUATLO
1137,iter5,Mitja Marató - Mollerussa - MM,2014-10-19,Domingo,21.097,road running,Absoluta/General,65,456,MOLLERUSSA,Pla d'Urgell,Lleida,RUN_MM
831,iter5,Infantils Pujada Seu Vella - Lleida - BEN-M,2022-12-18,Domingo,0.000,road running,Infantil,0,7,LLEIDA,Segrià,Lleida,INFANTILS
400,iter5,Cros Ciutat de Mollerussa - SUB 20 M,2021-02-28,Domingo,0.000,road running,Absoluta/General,0,10,MOLLERUSSA,Pla d'Urgell,Lleida,CROS
197,iter5,Cirera - Corbins - 10K,2016-05-29,Domingo,10.000,road running,Absoluta/General,37,183,CORBINS,Segrià,Lleida,RUN_10
26,iter5,Agro-Llobera Sant Blai - Palau Anglesola - 5K,2017-01-29,Domingo,5.000,road running,Absoluta/General,76,69,PALAU ANGLESOLA,Pla d'Urgell,Lleida,RUN_5


In [10]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/iter5/DF_ITER5_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\iter5_data\DF_ITER5_LIMPIO.csv
